In [1]:
import numpy as np
import pandas as pd
import polpo.preprocessing.dict as ppdict
import polpo.preprocessing.pd as ppd
import tabulate
from polpo.model_eval import (
    MeshEuclideanR2Score,
    MeshR2Score,
    MultiEvaluator,
    OlsPValues,
    R2Score,
    ReconstructionError,
    ResultsExtender,
    VertexReconstructionError,
    collect_obj_regr_eval_results,
)
from polpo.models import ObjectRegressor, SupervisedEmbeddingRegressor
from polpo.preprocessing import PartiallyInitializedStep
from polpo.preprocessing.learning import DictsToXY
from polpo.preprocessing.load.pregnancy import (
    DenseMaternalCsvDataLoader,
    DenseMaternalMeshLoader,
)
from polpo.preprocessing.mesh.conversion import PvFromData
from polpo.preprocessing.mesh.io import FreeSurferReader
from polpo.preprocessing.mesh.registration import PvAlign
from polpo.sklearn.adapter import AdapterPipeline, EvaluatedModel
from polpo.sklearn.mesh import BiMeshesToVertices
from polpo.sklearn.np import BiFlattenButFirst
from scipy.stats import f
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import FunctionTransformer

[KeOps] Warning : cuda was detected, but driver API could not be initialized. Switching to cpu only.


In [2]:
subject_id = "01"
pilot = subject_id == "01"

# Consistent structure names for display
structure_display_names = {
    "Hipp": "Hippocampus",
    "Amyg": "Amygdala",
    "Thal": "Thalamus-Proper",
    "Caud": "Caudate",
    "Puta": "Putamen",
    "Pall": "Pallidum",
    "Accu": "Accumbens",
}

# List of structures to loop through
structures = ["Thal", "Caud", "Puta", "Pall", "Hipp", "Amyg"]

n_structs = len(structures) * 2

In [3]:
# Load data
csv_loader = DenseMaternalCsvDataLoader(pilot=pilot, subject_id=subject_id)
df = csv_loader()

# Preprocess predictor
session_selector = ppd.DfIsInFilter("stage", ["post"], negate=True)
predictor_selector = (
    session_selector + ppd.ColumnsSelector("gestWeek") + ppd.SeriesToDict()
)
x_dict = predictor_selector(df)
# print(x_dict)
train_dict = {k: v for k, v in x_dict.items() if 1 <= k <= 16}
test_dict = {k: v for k, v in x_dict.items() if 17 <= k <= 19}

INFO: Data has already been downloaded... using cached file ('/home/luisfpereira/.herbrain/data/maternal/raw/28Baby_Hormones.csv').


In [4]:
# Model and pipeline setup
objs2y = AdapterPipeline(
    steps=[
        BiMeshesToVertices(index=0),
        FunctionTransformer(func=np.stack),
        BiFlattenButFirst(),
    ]
)

model = SupervisedEmbeddingRegressor(
    EvaluatedModel(
        PLSRegression(n_components=2),
        MultiEvaluator(
            [
                ReconstructionError(),
                VertexReconstructionError(prefix="vertex"),
            ]
        ),
    ),
    EvaluatedModel(
        LinearRegression(),
        MultiEvaluator([OlsPValues(), R2Score()]),
    ),
)

obj_model = EvaluatedModel(
    ObjectRegressor(model, objs2y),
    MultiEvaluator(
        [MeshEuclideanR2Score(), MeshR2Score()],
        extender=ResultsExtender(),
    ),
)

In [5]:
# Prepare an empty list to collect results
results_list = []
results_dict = {}

# Loop through each structure and hemisphere
for struct in structures:
    row_results = {}
    for left in [True, False]:
        try:
            # Load and preprocess meshes
            mesh_loader = DenseMaternalMeshLoader(
                subject_id=subject_id,
                as_dict=True,
                left=left,
                struct=struct,
                derivative="enigma",
            )
            mesh_reader = ppdict.DictMap(FreeSurferReader() + PvFromData())

            prep_pipe = PartiallyInitializedStep(
                Step=lambda **kwargs: ppdict.DictMap(PvAlign(**kwargs)),
                _target=lambda meshes: meshes[list(meshes.keys())[0]],
                max_iterations=500,
            )

            mesh_pipe = mesh_loader + mesh_reader + prep_pipe
            meshes = mesh_pipe()

            # Create dataset
            dataset_pipe = DictsToXY()
            X, meshes_ = dataset_pipe((x_dict, meshes))
            X_train, meshes_train = dataset_pipe((train_dict, meshes))
            X_test, meshes_test = dataset_pipe((test_dict, meshes))

            # Fit and evaluate
            # obj_model.fit(X, meshes_)
            obj_model.fit(X_train, meshes_train)
            predictions_test = obj_model.predict(X_test)

            # Evaluate on train and test sets
            eval_results = collect_obj_regr_eval_results(obj_model)

            # Flatten true and predicted meshes
            true_flat = np.stack([m.points.flatten() for m in meshes_test])
            pred_flat = np.stack([m.points.flatten() for m in predictions_test])

            # Compute global R² for test data
            r2_test = r2_score(true_flat.flatten(), pred_flat.flatten())

            # Extract p-values and R² from "regr-encoder"
            regr_encoder = eval_results["regr-encoder"]
            pvalues = eval_results["regr-regr"]["pvals"]
            r2_train = eval_results["obj_regr"]["featurewise_r2-mean"]
            r2_test = r2_test

            # Compute p-value for H0: R² = 0
            n = len(X_test)  # number of samples in test set
            k = 1  # single predictor (gestWeek)
            F_stat = (r2_test / k) / ((1 - r2_test) / (n - k - 1))
            p_value_r2 = 1 - f.cdf(F_stat, k, n - k - 1)

            # Format for display
            formatted_entry = f"p={pvalues[0].item():.3e}, R2(train)={r2_train:.4f}, R2(test)={r2_test:.4f}"
            row_results["left" if left else "right"] = formatted_entry

            # Store results
            results_list.append(
                {
                    "structure": struct,
                    "left": left,
                    "p-values": pvalues[0],
                    "adj-p-values": min(
                        eval_results["regr-regr"]["adj-pvals"][0] * n_structs, 1
                    ),
                    "r2-train": eval_results["regr-regr"]["r2"][0],
                    "r2-mesh-train": r2_train,
                    "r2-mesh-test": r2_test,
                    "r2-mesh-test-p-value": p_value_r2,
                }
            )

        except Exception as e:
            print(f"Error processing {struct} {'left' if left else 'right'}: {e}")
            results_list.append(
                {
                    "structure": struct,
                    "left": left,
                    "p-values": np.nan,
                    "r2-train": np.nan,
                    "r2-test": np.nan,
                    "r2-test-p-value": np.nan,
                }
            )

    # Store results with display name
    structure_name = structure_display_names.get(struct, struct)
    results_dict[structure_name] = row_results

# Convert to DataFrame for pretty display
final_df = pd.DataFrame.from_dict(results_dict, orient="index")
final_df.index.name = "Structure"
final_df.columns = ["Left", "Right"]

# Display final table
print(final_df)
final_df.to_csv("results_table_pls.csv")

# # Create a DataFrame for easier viewing
# results_df = pd.DataFrame(results_list)

                                                           Left  \
Structure                                                         
Thalamus-Proper  p=4.769e-07, R2(train)=0.1129, R2(test)=0.9991   
Caudate          p=8.868e-05, R2(train)=0.0904, R2(test)=0.9998   
Putamen          p=6.517e-05, R2(train)=0.1603, R2(test)=0.9995   
Pallidum         p=8.393e-07, R2(train)=0.2157, R2(test)=0.9991   
Hippocampus      p=1.489e-06, R2(train)=0.1183, R2(test)=0.9994   
Amygdala         p=1.227e-03, R2(train)=0.1466, R2(test)=0.9909   

                                                          Right  
Structure                                                        
Thalamus-Proper  p=7.031e-07, R2(train)=0.1260, R2(test)=0.9995  
Caudate          p=3.950e-05, R2(train)=0.1455, R2(test)=0.9997  
Putamen          p=2.025e-06, R2(train)=0.1438, R2(test)=0.9998  
Pallidum         p=1.356e-06, R2(train)=0.2179, R2(test)=0.9998  
Hippocampus      p=1.948e-06, R2(train)=0.1078, R2(test)=0.9997  
A

In [6]:
results_df = pd.DataFrame(results_list)

print(tabulate.tabulate(results_df, headers="keys", tablefmt="psql"))

+----+-------------+--------+-------------+----------------+------------+-----------------+----------------+------------------------+
|    | structure   | left   |    p-values |   adj-p-values |   r2-train |   r2-mesh-train |   r2-mesh-test |   r2-mesh-test-p-value |
|----+-------------+--------+-------------+----------------+------------+-----------------+----------------+------------------------|
|  0 | Thal        | True   | 4.76867e-07 |    1.14448e-05 |   0.845341 |       0.112902  |       0.999111 |             0.0189805  |
|  1 | Thal        | False  | 7.03128e-07 |    1.68751e-05 |   0.836623 |       0.125967  |       0.999451 |             0.014922   |
|  2 | Caud        | True   | 8.86767e-05 |    0.00212824  |   0.678057 |       0.0903885 |       0.999823 |             0.00847862 |
|  3 | Caud        | False  | 3.95002e-05 |    0.000948004 |   0.712333 |       0.145495  |       0.999652 |             0.0118788  |
|  4 | Puta        | True   | 6.51691e-05 |    0.00156406  |  